# ICVSP: pothole & speed-bump detector on Kaggle

Runs **unattended**: click **Save Version → Save & Run All (Commit)**, close the browser, and come back later.
Everything in `/kaggle/working` (the trained model, metrics, plots and `icvsp_results.zip`) is kept in the version's **Output** tab.

**One-time setup** (right-hand panel → *Session options*):
- **Accelerator: GPU T4 x2**
- **Internet: On** (needs a phone-verified Kaggle account; used to install packages and download the public SBP-YOLO dataset)

Limits: about 30 GPU hours per week, 12 hours per run. v1 (0.825) and v2 (0.837) are done.
**Next run: v3**, which fine-tunes v2 on Egyptian / night frames plus synthetic night copies (about 3–4 h).

**v3 needs your labelled frames as a private Kaggle dataset** (never public: YouTube frames, faces, plates):
1. kaggle.com → *Datasets → New Dataset*, name it **icvsp-egypt-train**, keep it **Private**.
2. Upload the zips, one per batch, **named by kind**, because the name becomes the source folder:
   - `egypt_<name>.zip` or `youtube_<name>.zip`: CVAT exports (*Ultralytics YOLO Detection 1.0, with images*)
   - `background_<name>.zip`: hazard-free night frames (reflections, wet road); just images, no labels
   - `v2_e150_960.pt` from `ai/models/`: the starting weights (needed while the GitHub repo is private)
3. In this notebook: *Add Input → Datasets → Your Datasets → icvsp-egypt-train*.

_Generated by `scripts/build_notebook.py`. Edit the files in `ICVSP/ai/` and regenerate; do not edit the embedded copies below._

## Settings: the only cell to edit
- `EXPERIMENT`: which run to train.
- `RESUME_FROM`: leave empty for a fresh run. To continue a run that was cut off, attach the earlier version's output
  (*Add Input → Your Work → this notebook*) and paste the path to its `last.pt`, e.g.
  `/kaggle/input/<notebook-name>/ai/runs/v1_e150/weights/last.pt`.

In [ ]:
EXPERIMENT = 'v3b_egypt_night' # done: v1 0.825, v2 0.837, v3_egypt_night 0.842 (mAP50, public test)
RESUME_FROM = ''                # '' = fresh run, or the path to an interrupted run's last.pt
NIGHT_SHARE = 0.3               # v3: share of daytime train images that get a synthetic night copy (0 = off)

EXPERIMENTS = {
    'v1_e150':        '--batch 32 --epochs 150 --patience 30',
    'v2_e150_960':    '--batch 16 --epochs 150 --patience 30 --imgsz 960',
    # fine-tune: starts from the v2 weights, so fewer epochs are needed
    'v3_egypt_night': '--model models/v2_e150_960.pt --batch 16 --epochs 80 --patience 20 --imgsz 960',
    # same recipe as v3, with the re-done CVAT boxes (27 bumps instead of 11) and the pothole re-check
    'v3b_egypt_night': '--model models/v2_e150_960.pt --batch 16 --epochs 80 --patience 20 --imgsz 960',
}
assert EXPERIMENT in EXPERIMENTS, f'unknown experiment {EXPERIMENT}'
ARGS = EXPERIMENTS[EXPERIMENT]
IMGSZ = 640 if EXPERIMENT == 'v1_e150' else 960

## 1. GPU and project folder

In [ ]:
!nvidia-smi -L
%mkdir -p /kaggle/working/ai/scripts /kaggle/working/ai/configs /kaggle/working/ai/runs
%cd /kaggle/working/ai

## 2. Project scripts

In [ ]:
%%writefile /kaggle/working/ai/requirements.txt
ultralytics>=8.3
requests
pyyaml
pillow
matplotlib


In [ ]:
%%writefile /kaggle/working/ai/configs/classes.yaml
# Unified class scheme for the ICVSP road-hazard detector.
# Every source dataset is remapped onto these IDs by scripts/merge_datasets.py.
classes:
  0: pothole
  1: speed_bump

# Source class names are matched case-insensitively against these regexes.
# A source class that matches nothing is dropped (its boxes are removed, the image is kept).
mapping:
  pothole: 'pothole|pot[_ -]?hole|d40'
  speed_bump: 'speed[_ -]?(bump|breaker|hump)|^bump|hump|breaker|speedbump'

# Classes that must never be mapped even if a regex above matches them.
exclude: 'rumble|crack|d00|d10|d20|manhole'

# Class names for sources that ship without a data.yaml, keyed by folder name under data/raw/.
sources:
  # SBP-YOLO has no data.yaml. Roboflow exports number classes alphabetically, so 0 = pothole,
  # 1 = speed bump is the likely order. VERIFY with the "check class IDs" cell in the notebook.
  sbp_yolo:
    names: [pothole, speed_bump]

# Sources made of video frames are split by video, never by frame (see merge_datasets.py).
# background_* folders hold hazard-free images (e.g. night reflections) and need no labels.
video_sources: '^(egypt|youtube|background)'
# Share of those videos used for validation (chosen by video name). 0 while there are only a few
# videos, so none of the scarce Egyptian labels is lost; raise to 0.15 with 10+ videos.
video_val_share: 0


In [ ]:
%%writefile /kaggle/working/ai/scripts/download_sbp.py
"""Download the public SBP-YOLO dataset without signing in to Google.

The authors share it as a public Google Drive folder of loose files (about 14,750 images and
labels), which is too many for `gdown --folder` (50-file limit). This script lists the folder
through Drive's public embedded view and downloads each file in parallel, with retries.
Files that already exist are skipped, so it is safe to re-run after an interruption.

Dataset: Liang et al., "SBP-YOLO", J. Real-Time Image Processing 23, 52 (2026).
         https://github.com/chuanqi1997/SBP-YOLO

Usage:  python scripts/download_sbp.py [--out /content/raw/sbp_yolo] [--workers 16]
"""
import argparse, re, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests

DATASET_FOLDER = '1anMQJlIeVCxH2xsk-cGJwVRb166ztyyJ'  # SBP-YOLO-Dataset/dataset (train/valid/test)
ENTRY = re.compile(r'<a href="https://drive.google.com/(drive/folders|file/d)/([^/"?]+)[^"]*"[^>]*>.*?'
                   r'flip-entry-title">([^<]+)', re.S)
session = requests.Session()


def list_folder(fid, prefix=''):
    html = session.get(f'https://drive.google.com/embeddedfolderview?id={fid}', timeout=60).text
    files = []
    for kind, item_id, name in ENTRY.findall(html):
        if kind == 'drive/folders':
            files += list_folder(item_id, f'{prefix}{name}/')
        else:
            files.append((f'{prefix}{name}', item_id))
    return files


def fetch(rel, fid, out: Path, tries=6):
    dst = out / rel
    if dst.exists() and dst.stat().st_size > 0:
        return 'skipped'
    dst.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(tries):
        try:
            r = session.get('https://drive.usercontent.google.com/download',
                            params={'id': fid, 'export': 'download'}, timeout=60)
            if r.status_code == 200 and not r.headers.get('content-type', '').startswith('text/html'):
                dst.write_bytes(r.content)
                return 'ok'
        except requests.RequestException:
            pass
        time.sleep(2 ** attempt)  # back off on rate limits (429) and transient errors
    return 'failed'


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--out', default='/content/raw/sbp_yolo', type=Path)
    ap.add_argument('--workers', type=int, default=16)
    a = ap.parse_args()

    print('Listing the public folder ...')
    files = list_folder(DATASET_FOLDER)
    print(f'{len(files)} files found; downloading with {a.workers} workers')
    stats, failed = {'ok': 0, 'skipped': 0, 'failed': 0}, []
    with ThreadPoolExecutor(a.workers) as pool:
        jobs = {pool.submit(fetch, rel, fid, a.out): rel for rel, fid in files}
        for n, job in enumerate(as_completed(jobs), 1):
            res = job.result()
            stats[res] += 1
            if res == 'failed':
                failed.append(jobs[job])
            if n % 1000 == 0 or n == len(files):
                print(f'  {n}/{len(files)}  {stats}')
    if failed:
        (a.out / 'failed.txt').write_text('\n'.join(failed))
        print(f'{len(failed)} files failed (listed in failed.txt). Re-run this script to retry them.')
    else:
        print('All files downloaded.')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/ai/scripts/merge_datasets.py
"""Merge YOLO-format road-hazard datasets into one dataset with the unified ICVSP classes.

For every source folder under data/raw/<source>/ that contains a data.yaml (or a classes.txt),
the script:
  1. remaps source class IDs to the unified scheme in configs/classes.yaml,
  2. drops boxes whose class does not map (e.g. rumble strips, cracks),
  3. removes near-duplicate images across all sources (perceptual dHash),
  4. writes data/merged/{train,val,test}/{images,labels} and data/merged/data.yaml.

Splits: a source's own test split is kept as test, so results stay comparable with the
source paper; everything else is re-split train/val 85/15 per source.

Video-derived sources (folder name matching `video_sources` in classes.yaml, e.g. egypt_*,
youtube_*, background_*) are split by VIDEO, not by frame: neighbouring frames of one drive are
near-copies, so splitting them by frame would leak the validation set into training.
With few videos, `video_val_share: 0` keeps every video in train: the public val set drives early
stopping, and the held-out test videos in data/videos/videos.csv measure the Egyptian result.
A frame's video is its file name without the trailing frame number (<video>_0042.jpg).

Background sources (folder name starting with background_) hold images with no hazards,
e.g. wet roads and streetlight reflections. They need no labels: every image gets an empty
label file, which teaches the model what is NOT a bump.

Usage:  python scripts/merge_datasets.py [--raw data/raw] [--out data/merged]
"""
import argparse, hashlib, json, random, re, shutil
from collections import Counter, defaultdict
from pathlib import Path

import yaml
from PIL import Image

ROOT = Path(__file__).resolve().parents[1]
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def load_scheme():
    cfg = yaml.safe_load((ROOT / 'configs/classes.yaml').read_text())
    names = [cfg['classes'][i] for i in sorted(cfg['classes'])]
    rules = [(names.index(k), re.compile(v, re.I)) for k, v in cfg['mapping'].items()]
    return names, rules, re.compile(cfg['exclude'], re.I)


def source_names(src: Path):
    if src.name.lower().startswith('background'):
        return []
    override = (yaml.safe_load((ROOT / 'configs/classes.yaml').read_text()).get('sources') or {}).get(src.name)
    if override:
        return list(override['names'])
    for y in list(src.rglob('data.yaml')) + list(src.rglob('*.yaml')):
        try:
            d = yaml.safe_load(y.read_text())
        except Exception:
            continue
        if isinstance(d, dict) and 'names' in d:
            n = d['names']
            return [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)
    for t in src.rglob('classes.txt'):
        return [l.strip() for l in t.read_text().splitlines() if l.strip()]
    raise SystemExit(f'No data.yaml / classes.txt with class names found in {src}')


def build_map(names, rules, exclude):
    m = {}
    for i, n in enumerate(names):
        if exclude.search(n):
            continue
        for uid, rx in rules:
            if rx.search(n):
                m[i] = uid
                break
    return m


def dhash(path, size=8):
    with Image.open(path) as im:
        im = im.convert('L').resize((size + 1, size))
        px = im.tobytes()  # one byte per pixel in 'L' mode
    bits = ''.join('1' if px[r * (size + 1) + c] > px[r * (size + 1) + c + 1] else '0'
                   for r in range(size) for c in range(size))
    return int(bits, 2)


def video_of(img: Path):
    return re.sub(r'_\d+$', '', img.stem)


def split_of(img: Path):
    parts = {p.lower() for p in img.parts}
    if 'test' in parts:
        return 'test'
    if 'valid' in parts or 'val' in parts:
        return 'val'
    return 'train'


def label_for(img: Path):
    # YOLO convention: .../images/x.jpg <-> .../labels/x.txt ; fall back to same folder
    parts = list(img.parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == 'images':
            parts[i] = 'labels'
            cand = Path(*parts).with_suffix('.txt')
            if cand.exists():
                return cand
    cand = img.with_suffix('.txt')
    return cand if cand.exists() else None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--raw', default=ROOT / 'data/raw', type=Path)
    ap.add_argument('--out', default=ROOT / 'data/merged', type=Path)
    ap.add_argument('--seed', default=0, type=int)
    args = ap.parse_args()
    random.seed(args.seed)
    names, rules, exclude = load_scheme()
    cfg = yaml.safe_load((ROOT / 'configs/classes.yaml').read_text())
    video_rx = re.compile(cfg.get('video_sources', '$^'), re.I)
    video_val_share = float(cfg.get('video_val_share', 0))

    if args.out.exists():
        shutil.rmtree(args.out)
    for s in ('train', 'val', 'test'):
        (args.out / s / 'images').mkdir(parents=True)
        (args.out / s / 'labels').mkdir(parents=True)

    seen = {}
    report = {'sources': {}, 'duplicates_removed': 0}
    counts = defaultdict(Counter)

    for src in sorted(p for p in args.raw.iterdir() if p.is_dir()):
        src_names = source_names(src)
        cmap = build_map(src_names, rules, exclude)
        imgs = sorted(p for p in src.rglob('*') if p.suffix.lower() in IMG_EXT)
        rep = {'source_classes': src_names, 'mapping': {src_names[k]: names[v] for k, v in cmap.items()},
               'images': 0, 'kept': 0, 'dropped_boxes': 0}
        pending = []
        for img in imgs:
            rep['images'] += 1
            try:
                h = dhash(img)
            except Exception:
                continue
            if h in seen:
                report['duplicates_removed'] += 1
                continue
            seen[h] = img
            lab = label_for(img)
            lines = []
            if lab:
                for ln in lab.read_text().splitlines():
                    f = ln.split()
                    if len(f) < 5:
                        continue
                    cid = int(float(f[0]))
                    if cid in cmap:
                        lines.append(' '.join([str(cmap[cid])] + f[1:5]))
                    else:
                        rep['dropped_boxes'] += 1
            pending.append((img, lines, split_of(img)))

        # re-split non-test images 85/15 per source (per video for video-derived sources)
        rest = [x for x in pending if x[2] != 'test']
        if video_rx.search(src.name):
            # choose val videos by a hash of the video name, so one video lands on the same side in
            # every zip it appears in (Y frames, backgrounds, CVAT fixes)
            val = lambda i: int(hashlib.md5(video_of(i).encode()).hexdigest(), 16) % 1000 < 1000 * video_val_share
            final = [(i, l, 'val' if val(i) else 'train') for i, l, _ in rest]
            vids = {video_of(i) for i, _, _ in rest}
            rep['videos'] = {'train': sorted(v for v in vids if not val(Path(v + '_0'))),
                             'val': sorted(v for v in vids if val(Path(v + '_0')))}
        else:
            random.shuffle(rest)
            n_val = round(0.15 * len(rest))
            final = [(i, l, 'val') for i, l, _ in rest[:n_val]] + [(i, l, 'train') for i, l, _ in rest[n_val:]]
        final += [x for x in pending if x[2] == 'test']

        tag = re.sub(r'[^a-z0-9]+', '_', src.name.lower())
        for img, lines, split in final:
            stem = f'{tag}_{hashlib.md5(str(img).encode()).hexdigest()[:10]}'
            shutil.copy2(img, args.out / split / 'images' / f'{stem}{img.suffix.lower()}')
            (args.out / split / 'labels' / f'{stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))
            for l in lines:
                counts[split][names[int(l.split()[0])]] += 1
            counts[split]['images'] += 1
            rep['kept'] += 1
        report['sources'][src.name] = rep

    (args.out / 'data.yaml').write_text(yaml.safe_dump({
        'path': str(args.out), 'train': 'train/images', 'val': 'val/images', 'test': 'test/images',
        'names': {i: n for i, n in enumerate(names)}}, sort_keys=False))
    report['counts'] = {k: dict(v) for k, v in counts.items()}
    (args.out / 'merge_report.json').write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/ai/scripts/train.py
"""Train the ICVSP pothole / speed-bump detector on the merged dataset.

Defaults are sized for a 16 GB Apple-silicon laptop (MPS). On Colab, pass --device 0 --batch 32.

Usage:  python scripts/train.py [--model yolo11n.pt] [--epochs 50] [--fraction 1.0]
        --fraction < 1.0 trains on a random subset (used for the learning curve).
        --resume runs/<name>/weights/last.pt continues an interrupted run where it stopped
        (same dataset path and runs folder as the original run, e.g. a later Kaggle session).
"""
import argparse
from pathlib import Path

import torch
from ultralytics import YOLO

ROOT = Path(__file__).resolve().parents[1]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', default=ROOT / 'data/merged/data.yaml')
    ap.add_argument('--model', default='yolo11n.pt')
    ap.add_argument('--epochs', type=int, default=50)
    ap.add_argument('--imgsz', type=int, default=640)
    ap.add_argument('--batch', type=int, default=16)
    ap.add_argument('--fraction', type=float, default=1.0)
    ap.add_argument('--device', default='mps' if torch.backends.mps.is_available() else 'cpu')
    ap.add_argument('--patience', type=int, default=15, help='stop early after this many epochs without improvement')
    ap.add_argument('--name', default=None)
    ap.add_argument('--resume', default=None, help="an interrupted run's last.pt; all other options come from it")
    a = ap.parse_args()

    if a.resume:
        YOLO(a.resume).train(resume=True)
        return

    name = a.name or f'{Path(a.model).stem}_e{a.epochs}_f{int(a.fraction * 100)}'
    YOLO(a.model).train(
        data=str(a.data), epochs=a.epochs, imgsz=a.imgsz, batch=a.batch, device=a.device,
        fraction=a.fraction, project=str(ROOT / 'runs'), name=name, exist_ok=True,
        patience=a.patience, workers=4, seed=0, plots=True,
    )


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/ai/scripts/evaluate.py
"""Evaluate a trained model and apply the "is the merge enough?" decision rule.

Evaluates on the public test split and, when it exists, on the Egyptian test set
(data/egypt_test/data.yaml). The decision is taken on the EGYPTIAN set only; the public
number is a sanity check.

Decision rule (agreed before testing, see proposal §AI plan):
  speed_bump recall >= 0.75 and precision >= 0.70  -> merge is enough
  0.50 <= recall < 0.75                            -> targeted top-up of the failure cases
  recall < 0.50                                    -> build a proper Egyptian training set

Usage:  python scripts/evaluate.py --weights runs/<run>/weights/best.pt
"""
import argparse, json
from pathlib import Path

import torch
from ultralytics import YOLO

ROOT = Path(__file__).resolve().parents[1]
ENOUGH_RECALL, ENOUGH_PRECISION, TOPUP_RECALL = 0.75, 0.70, 0.50


def per_class(metrics, names):
    out = {}
    present = list(metrics.box.ap_class_index)  # class_result() indexes by position in this list
    for i, n in names.items():
        p, r, map50, map5095 = metrics.box.class_result(present.index(i)) if i in present else (0, 0, 0, 0)
        out[n] = {'precision': round(float(p), 3), 'recall': round(float(r), 3),
                  'mAP50': round(float(map50), 3), 'mAP50-95': round(float(map5095), 3)}
    return out


def verdict(cls):
    sb = cls.get('speed_bump', {})
    r, p = sb.get('recall', 0), sb.get('precision', 0)
    if r >= ENOUGH_RECALL and p >= ENOUGH_PRECISION:
        return 'ENOUGH: public merge generalises to Egyptian speed bumps'
    if r >= TOPUP_RECALL:
        return 'TOP-UP: collect 50-100 images of the failure cases only'
    return 'NOT ENOUGH: build a proper Egyptian training set'


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--weights', required=True)
    ap.add_argument('--device', default='mps' if torch.backends.mps.is_available() else 'cpu')
    ap.add_argument('--public-data', default=ROOT / 'data/merged/data.yaml', type=Path)
    ap.add_argument('--egypt-data', default=ROOT / 'data/egypt_test/data.yaml', type=Path)
    ap.add_argument('--imgsz', type=int, default=640, help='use the same size the model was trained at')
    a = ap.parse_args()
    model = YOLO(a.weights)
    results = {}
    sets = {'public_test': a.public_data, 'egypt_test': a.egypt_data}
    for tag, yml in sets.items():
        if not yml.exists():
            results[tag] = 'missing'
            continue
        run = Path(a.weights).parents[1].name
        m = model.val(data=str(yml), split='test', device=a.device, plots=True, imgsz=a.imgsz,
                      project=str(ROOT / 'runs'), name=f'eval_{tag}_{run}', exist_ok=True)
        results[tag] = {'overall_mAP50': round(float(m.box.map50), 3), 'per_class': per_class(m, model.names)}
    if isinstance(results.get('egypt_test'), dict):
        results['decision'] = verdict(results['egypt_test']['per_class'])
    else:
        results['decision'] = 'PENDING: no Egyptian test set yet (see README, step 3)'
    out = Path(a.weights).parents[1] / 'evaluation.json'
    out.write_text(json.dumps(results, indent=2))
    print(json.dumps(results, indent=2))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/ai/scripts/predict_clips.py
"""Quick visual check: run the detector on the team's Egyptian phone clips.

Saves each clip with the model's boxes drawn on it, plus a per-clip summary of when it saw a
speed bump or pothole. This shows WHAT the model detects, but it is not a measurement:
it cannot tell you what the model missed. For recall/precision numbers, label the frames
and use evaluate.py (README, step 3).

Also writes detections.csv (one row per box: clip, time, class, confidence, box corners as
fractions of the frame), which event_recall.py matches against the team's list of bumps and
potholes, and tools/review.html draws over the video.

Usage:  python scripts/predict_clips.py --weights models/v1_e150.pt [--conf 0.25] [--stride 2]
"""
import argparse, csv
from pathlib import Path

import cv2
import torch
from ultralytics import YOLO

ROOT = Path(__file__).resolve().parents[1]
VIDEO = {'.mov', '.mp4', '.m4v', '.avi'}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--weights', required=True)
    ap.add_argument('--clips', default=ROOT / 'data/egypt_test/clips', type=Path)
    ap.add_argument('--out', default=ROOT / 'runs/egypt_clips', type=Path)
    ap.add_argument('--conf', type=float, default=0.25)
    ap.add_argument('--imgsz', type=int, default=640)
    ap.add_argument('--stride', type=int, default=1, help='check every n-th frame (2 = half the frames)')
    ap.add_argument('--no-video', action='store_true', help='skip the annotated video copy (faster; review.html draws the boxes)')
    ap.add_argument('--device', default='0' if torch.cuda.is_available() else
                    'mps' if torch.backends.mps.is_available() else 'cpu')
    a = ap.parse_args()

    model = YOLO(a.weights)
    rows, dets = [], []
    for clip in sorted(p for p in a.clips.iterdir() if p.suffix.lower() in VIDEO):
        hits = {n: [] for n in model.names.values()}
        cap = cv2.VideoCapture(str(clip)); fps = cap.get(cv2.CAP_PROP_FPS) or 30.0; cap.release()
        frames = 0
        for r in model.predict(source=str(clip), conf=a.conf, imgsz=a.imgsz, device=a.device, stream=True,
                               vid_stride=a.stride, save=not a.no_video, project=str(a.out), name=clip.stem,
                               exist_ok=True, verbose=False):
            t = frames * a.stride / fps
            for c, s, (x1, y1, x2, y2) in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xyxyn.tolist()):
                hits[model.names[int(c)]].append((t, s))
                dets.append({'clip': clip.name, 't_s': round(t, 3), 'class': model.names[int(c)], 'conf': round(s, 3),
                             'x1': round(x1, 4), 'y1': round(y1, 4), 'x2': round(x2, 4), 'y2': round(y2, 4)})
            frames += 1
        for cls, h in hits.items():
            rows.append({'clip': clip.name, 'class': cls, 'frames_with_detection': len({round(t, 3) for t, _ in h}),
                         'first_seen_s': round(min(t for t, _ in h), 1) if h else '',
                         'last_seen_s': round(max(t for t, _ in h), 1) if h else '',
                         'max_confidence': round(max(s for _, s in h), 2) if h else ''})
        seen = ', '.join(f'{c}: {len(h)} boxes' for c, h in hits.items())
        print(f'{clip.name}: {frames} frames checked ({frames * a.stride / fps:.0f} s) -> {seen}', flush=True)
    a.out.mkdir(parents=True, exist_ok=True)
    with open(a.out / 'summary.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]) if rows else ['clip'])
        w.writeheader(); w.writerows(rows)
    with open(a.out / 'detections.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['clip', 't_s', 'class', 'conf', 'x1', 'y1', 'x2', 'y2'])
        w.writeheader(); w.writerows(dets)
    print(f'\nAnnotated videos, summary.csv and detections.csv saved in {a.out}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/ai/scripts/extract_frames.py
"""Turn the team's short phone clips into frames for labelling.

Put clips in data/egypt_test/clips/<area>_<person>_<n>.mov (the area prefix is used to
split by location, never by frame). Frames go to data/egypt_test/frames/<clip-name>/.

Usage:  python scripts/extract_frames.py [--fps 2]
"""
import argparse, subprocess
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--fps', type=float, default=2)
    a = ap.parse_args()
    clips = sorted(p for p in (ROOT / 'data/egypt_test/clips').iterdir()
                   if p.suffix.lower() in {'.mov', '.mp4', '.m4v'})
    for clip in clips:
        out = ROOT / 'data/egypt_test/frames' / clip.stem
        out.mkdir(parents=True, exist_ok=True)
        subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', str(clip), '-vf', f'fps={a.fps}',
                        '-q:v', '2', str(out / f'{clip.stem}_%04d.jpg')], check=True)
        print(f'{clip.name}: {len(list(out.glob("*.jpg")))} frames')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/ai/scripts/prelabel.py
"""Pre-label the Egyptian frames with the model, so labelling means correcting boxes, not drawing them.

Reads every frame under data/egypt_test/frames/, runs the detector, and writes an
Ultralytics-YOLO dataset that CVAT can import ("Ultralytics YOLO Detection 1.0"):

  data/egypt_test/prelabels/
    data.yaml
    images/train/<frame>.jpg
    labels/train/<frame>.txt     (empty file = model saw nothing; still check the frame)

Usage:  python scripts/prelabel.py --weights models/v1_e150.pt [--conf 0.25]
"""
import argparse, shutil
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

ROOT = Path(__file__).resolve().parents[1]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--weights', required=True)
    ap.add_argument('--frames', default=ROOT / 'data/egypt_test/frames', type=Path)
    ap.add_argument('--out', default=ROOT / 'data/egypt_test/prelabels', type=Path)
    ap.add_argument('--conf', type=float, default=0.25)
    ap.add_argument('--imgsz', type=int, default=640)
    ap.add_argument('--device', default='0' if torch.cuda.is_available() else
                    'mps' if torch.backends.mps.is_available() else 'cpu')
    a = ap.parse_args()

    frames = sorted(a.frames.rglob('*.jpg'))
    if not frames:
        raise SystemExit(f'No frames in {a.frames}. Run scripts/extract_frames.py first.')
    if a.out.exists():
        shutil.rmtree(a.out)
    img_dir, lab_dir = a.out / 'images/train', a.out / 'labels/train'
    img_dir.mkdir(parents=True)
    lab_dir.mkdir(parents=True)

    model = YOLO(a.weights)
    boxes = {n: 0 for n in model.names.values()}
    for f in frames:
        shutil.copy2(f, img_dir / f.name)
        r = model.predict(source=str(f), conf=a.conf, imgsz=a.imgsz, device=a.device, verbose=False)[0]
        lines = []
        for c, (x, y, w, h) in zip(r.boxes.cls.tolist(), r.boxes.xywhn.tolist()):
            lines.append(f'{int(c)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
            boxes[model.names[int(c)]] += 1
        (lab_dir / f'{f.stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))

    (a.out / 'data.yaml').write_text(yaml.safe_dump(
        {'path': '.', 'train': 'images/train', 'names': dict(model.names)}, sort_keys=False))
    (a.out / 'train.txt').write_text(''.join(f'images/train/{f.name}\n' for f in frames))
    print(f'{len(frames)} frames pre-labelled: ' + ', '.join(f'{n} {k}' for n, k in boxes.items()))
    print(f'Import {a.out} into CVAT as "Ultralytics YOLO Detection 1.0", correct every box, add the missed ones.')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/ai/scripts/prepare_egypt_test.py
"""Turn the team's corrected labels into the Egyptian test set used by evaluate.py.

Takes a YOLO-format export (a folder or .zip, e.g. CVAT "Ultralytics YOLO Detection 1.0"),
collects every labelled image regardless of how the export split it, maps classes by NAME
onto the ICVSP classes, and writes

  data/egypt_test/test/{images,labels}/
  data/egypt_test/data.yaml

It also reports how many objects came from each area (the part of the file name before the
first "_", e.g. maadi_malak_1_0003.jpg → maadi), because the decision needs bumps from
several different places.

Usage:  python scripts/prepare_egypt_test.py path/to/export(.zip)
"""
import argparse, shutil, tempfile, zipfile
from collections import Counter, defaultdict
from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parents[1]
IMG_EXT = {'.jpg', '.jpeg', '.png'}
TARGET_SPEED_BUMPS = 120


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('export', type=Path)
    ap.add_argument('--out', default=ROOT / 'data/egypt_test', type=Path)
    a = ap.parse_args()

    ours = yaml.safe_load((ROOT / 'configs/classes.yaml').read_text())['classes']
    our_id = {name: i for i, name in ours.items()}

    tmp = None
    src = a.export
    if src.suffix == '.zip':
        tmp = Path(tempfile.mkdtemp())
        zipfile.ZipFile(src).extractall(tmp)
        src = tmp
    ymls = [y for y in src.rglob('*.yaml') if 'names' in (yaml.safe_load(y.read_text()) or {})]
    if not ymls:
        raise SystemExit('No data.yaml with class names found in the export.')
    names = yaml.safe_load(ymls[0].read_text())['names']
    names = names if isinstance(names, dict) else dict(enumerate(names))
    remap = {}
    for i, n in names.items():
        key = str(n).strip().lower().replace(' ', '_').replace('-', '_')
        if key not in our_id:
            raise SystemExit(f'Export class "{n}" is not one of {list(our_id)}. Rename it in the labelling tool.')
        remap[int(i)] = our_id[key]

    test = a.out / 'test'
    if test.exists():
        shutil.rmtree(test)
    (test / 'images').mkdir(parents=True)
    (test / 'labels').mkdir(parents=True)

    per_class, per_area, images, unlabelled = Counter(), defaultdict(Counter), 0, 0
    for img in sorted(p for p in src.rglob('*') if p.suffix.lower() in IMG_EXT):
        parts = list(img.parts)
        lab = None
        for i in range(len(parts) - 1, -1, -1):
            if parts[i] == 'images':
                lab = Path(*parts[:i], 'labels', *parts[i + 1:]).with_suffix('.txt')
                break
        if lab is None or not lab.exists():
            unlabelled += 1
            continue
        lines = []
        for ln in lab.read_text().splitlines():
            f = ln.split()
            if len(f) >= 5:
                c = remap[int(float(f[0]))]
                lines.append(' '.join([str(c)] + f[1:5]))
                per_class[ours[c]] += 1
                per_area[img.stem.split('_')[0]][ours[c]] += 1
        shutil.copy2(img, test / 'images' / img.name)
        (test / 'labels' / f'{img.stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))
        images += 1

    (a.out / 'data.yaml').write_text(yaml.safe_dump(
        {'path': str(a.out.resolve()), 'train': 'test/images', 'val': 'test/images', 'test': 'test/images',
         'names': ours}, sort_keys=False))
    if tmp:
        shutil.rmtree(tmp)

    print(f'Egyptian test set: {images} images ({unlabelled} skipped without a label file)')
    print('  objects: ' + ', '.join(f'{k} {v}' for k, v in per_class.items()))
    print('  by area: ' + '; '.join(f'{area}: ' + ', '.join(f'{k} {v}' for k, v in c.items())
                                     for area, c in sorted(per_area.items())))
    sb = per_class.get('speed_bump', 0)
    if sb < TARGET_SPEED_BUMPS or len(per_area) < 5:
        print(f'  ⚠ Target is about {TARGET_SPEED_BUMPS} speed bumps from at least 5 areas '
              f'(have {sb} from {len(per_area)}). Results will be less certain.')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/ai/scripts/make_night.py
"""Add synthetic night copies of daytime training images to the merged dataset.

Almost all public pothole / speed-bump images are daytime. For a share of the daytime TRAINING
images (never val or test), this writes a night version next to the original, with the same labels:

  - darker image with a sodium-orange or LED-blue cast, black sky, brighter near the headlights,
  - wet-road reflections of lamps and tail lights (vertical coloured streaks) and a glossy band
    across the road, always AWAY from the labelled boxes. They carry no label, so the model learns
    that a bright streak is not a speed bump (the failure seen on the Cairo night video at 4:03),
  - sensor noise and slight blur, as in a phone or dashcam at night.

Images that are already dark (mean brightness below --dark) are left alone.
Night copies are named <original>_night.jpg, so a re-run replaces them instead of adding more.

Usage:  python scripts/make_night.py [--data data/merged] [--share 0.3]
        python scripts/make_night.py --preview 12     # writes a contact sheet to check the look
"""
import argparse, random
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter

ROOT = Path(__file__).resolve().parents[1]


def boxes_of(label: Path, w, h):
    out = []
    if label.exists():
        for ln in label.read_text().splitlines():
            f = ln.split()
            if len(f) >= 5:
                x, y, bw, bh = (float(v) for v in f[1:5])
                out.append(((x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h))
    return out


def _free(x0, y0, x1, y1, boxes):
    return not any(x0 < bx1 and x1 > bx0 and y0 < by1 and y1 > by0 for bx0, by0, bx1, by1 in boxes)


def night(img: Image.Image, boxes, rng: random.Random):
    """Modelled on the Cairo night video: lit streets, dark sky, glossy wet road."""
    a = np.asarray(img.convert('RGB')).astype(np.float32) / 255
    h, w = a.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w]
    a = a ** rng.uniform(1.4, 2.0) * rng.uniform(0.45, 0.75)            # darker, crushed shadows
    a *= np.array([0.95, 0.9, 0.85]) if rng.random() < 0.6 else np.array([0.85, 0.92, 1.05])  # sodium or LED light
    sky = np.clip((0.45 * h - yy) / (0.25 * h), 0, 1)[..., None]         # upper part: sky goes black
    a *= 1 - 0.8 * sky
    d = np.sqrt(((xx - w / 2) / (0.9 * w)) ** 2 + ((yy - 1.1 * h) / (1.0 * h)) ** 2)
    a *= np.clip(1.3 - d, 0.35, 1.0)[..., None]                          # headlights: brighter near the car
    colours = [np.array([1.0, 0.65, 0.3]), np.array([1.0, 0.25, 0.2]), np.array([1.0, 0.95, 0.85])]
    for _ in range(rng.randint(1, 5)):                                   # lamp / tail-light reflections:
        c = rng.choice(colours) * rng.uniform(0.3, 0.7)                  # vertical streaks on the road
        sw, sh = rng.uniform(0.01, 0.035) * w, rng.uniform(0.12, 0.35) * h
        for _try in range(10):
            x0, y0 = rng.uniform(0, w - sw), rng.uniform(0.55 * h, h - 0.5 * sh)
            if _free(x0, y0, x0 + sw, y0 + sh, boxes):
                a += np.exp(-(((xx - x0 - sw / 2) / (sw / 2)) ** 2 + ((yy - y0 - sh / 2) / (sh / 2)) ** 2))[..., None] * c
                break
    if rng.random() < 0.6:                                               # glossy wet band across the road,
        sw, sh = rng.uniform(0.25, 0.6) * w, rng.uniform(0.03, 0.08) * h # where the false bump boxes landed
        for _try in range(10):
            x0, y0 = rng.uniform(0, w - sw), rng.uniform(0.55 * h, 0.9 * h)
            if _free(x0, y0, x0 + sw, y0 + sh, boxes):
                g = np.exp(-(((xx - x0 - sw / 2) / (sw / 2)) ** 4 + ((yy - y0 - sh / 2) / (sh / 2)) ** 2))
                a += g[..., None] * rng.choice(colours) * rng.uniform(0.08, 0.2)
                break
    a += np.random.default_rng(rng.randint(0, 2**31)).normal(0, rng.uniform(0.01, 0.025), a.shape)
    out = Image.fromarray((np.clip(a, 0, 1) * 255).astype(np.uint8))
    return out.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 0.8)))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', default=ROOT / 'data/merged', type=Path)
    ap.add_argument('--share', type=float, default=0.3, help='share of daytime train images that get a night copy')
    ap.add_argument('--dark', type=float, default=70, help='skip images whose mean brightness (0-255) is below this')
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--preview', type=int, default=0, help='only write a contact sheet of this many examples')
    a = ap.parse_args()
    rng = random.Random(a.seed)

    img_dir, lab_dir = a.data / 'train/images', a.data / 'train/labels'
    imgs = sorted(p for p in img_dir.iterdir() if '_night' not in p.stem and not p.stem.startswith('background'))
    rng.shuffle(imgs)
    picked = imgs[:a.preview] if a.preview else imgs[:round(a.share * len(imgs))]

    made, skipped, sheet = 0, 0, []
    for p in picked:
        im = Image.open(p).convert('RGB')
        if np.asarray(im.convert('L')).mean() < a.dark:
            skipped += 1
            continue
        lab = lab_dir / f'{p.stem}.txt'
        n = night(im, boxes_of(lab, *im.size), rng)
        if a.preview:
            sheet.append((im, n))
            continue
        n.save(img_dir / f'{p.stem}_night.jpg', quality=90)
        (lab_dir / f'{p.stem}_night.txt').write_text(lab.read_text() if lab.exists() else '')
        made += 1

    if a.preview:
        W = 480
        rows = [(o.resize((W, int(o.height * W / o.width))), n.resize((W, int(n.height * W / n.width)))) for o, n in sheet]
        H = sum(r[0].height for r in rows)
        out = Image.new('RGB', (2 * W, H), 'white')
        y = 0
        for o, n in rows:
            out.paste(o, (0, y)); out.paste(n, (W, y)); y += o.height
        dest = ROOT / 'runs/night_preview.jpg'
        dest.parent.mkdir(parents=True, exist_ok=True)
        out.save(dest, quality=85)
        print(f'Preview (left: original, right: night copy) saved to {dest}')
        return
    print(f'{made} night copies added to {img_dir} ({skipped} already-dark images skipped)')


if __name__ == '__main__':
    main()


## 3. Install packages

In [ ]:
!pip -q install -r /kaggle/working/ai/requirements.txt
import ultralytics; print("ultralytics", ultralytics.__version__, "installed")

## 3b. v3 only: check the Egyptian data and the v2 starting weights (fails in seconds, before the long download)

In [ ]:
import glob, pathlib, shutil, urllib.request
if EXPERIMENT.startswith('v3'):
    roots = glob.glob('/kaggle/input/**/icvsp-egypt-train', recursive=True)
    assert roots, 'Attach the private icvsp-egypt-train dataset: Add Input -> Your Datasets.'
    pathlib.Path('models').mkdir(exist_ok=True)
    found = glob.glob('/kaggle/input/**/v2_e150_960.pt', recursive=True)
    if found:                                   # uploaded into the private dataset (works with a private repo)
        shutil.copy(found[0], 'models/v2_e150_960.pt')
    else:                                       # fallback: the GitHub repo, only if it is public
        try:
            urllib.request.urlretrieve('https://github.com/S2800-0/icvsp-prototype/raw/main/ai/models/v2_e150_960.pt',
                                       'models/v2_e150_960.pt')
        except Exception as e:
            raise SystemExit('v2_e150_960.pt not found: upload ai/models/v2_e150_960.pt into the icvsp-egypt-train '
                             f'dataset (the GitHub repo is private or unreachable: {e})')
    print('v2 weights ready:', pathlib.Path('models/v2_e150_960.pt').stat().st_size // 1024, 'KB')
    print('Egyptian data:', sorted(x.name for r in roots for x in pathlib.Path(r).iterdir()))

## 4. Download SBP-YOLO, add the Egyptian / night data, merge
The data goes to `/tmp` so it is not saved into the output. Class IDs were verified on 29 Sep (0 = pothole, 1 = speed bump). The download is retried automatically if Drive rate-limits it. For v3, the zips from the private `icvsp-egypt-train` dataset are unpacked next to SBP-YOLO; video frames are split by video, never by frame.

In [ ]:
import subprocess
for attempt in range(3):
    subprocess.run(['python', 'scripts/download_sbp.py', '--out', '/tmp/raw/sbp_yolo'], check=True)
    if not __import__('pathlib').Path('/tmp/raw/sbp_yolo/failed.txt').exists():
        break
    __import__('os').remove('/tmp/raw/sbp_yolo/failed.txt')
import glob, pathlib, shutil, urllib.request, zipfile
# Kaggle unzips uploaded zips into folders, so accept both zips and folders, wherever the dataset is mounted
roots = [pathlib.Path(p) for p in glob.glob('/kaggle/input/**/icvsp-egypt-train', recursive=True)]
items = sorted(x for r in roots for x in r.iterdir()) if roots else []
for x in items:
    name = x.name.lower().removesuffix('.zip')
    if not name.startswith(('egypt', 'youtube', 'background')):
        print('skipped', x.name, '(name it egypt_*, youtube_* or background_*)'); continue
    if x.suffix.lower() == '.zip':
        zipfile.ZipFile(x).extractall(f'/tmp/raw/{name}')
    elif x.is_dir():
        shutil.copytree(x, f'/tmp/raw/{name}', dirs_exist_ok=True)
    print(name, sum(1 for _ in pathlib.Path(f'/tmp/raw/{name}').rglob('*.jpg')), 'images')
!python scripts/merge_datasets.py --raw /tmp/raw --out /tmp/merged
if EXPERIMENT.startswith('v3') and NIGHT_SHARE > 0:
    !python scripts/make_night.py --data /tmp/merged --share {NIGHT_SHARE}
!cp /tmp/merged/merge_report.json runs/

## 5. Train
A fresh run trains from the YOLO11n starting weights; a resumed run continues from `RESUME_FROM` with its original settings.

In [ ]:
import pathlib, shutil
if RESUME_FROM:
    last = pathlib.Path(f'runs/{EXPERIMENT}/weights/last.pt')
    last.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(RESUME_FROM, last)
    !python scripts/train.py --resume {last}
else:
    !python scripts/train.py --data /tmp/merged/data.yaml --device 0 {ARGS} --name {EXPERIMENT}

## 6. Evaluate on the public test set

In [ ]:
!python scripts/evaluate.py --weights runs/{EXPERIMENT}/weights/best.pt --device 0 --imgsz {IMGSZ} --public-data /tmp/merged/data.yaml

## 7. Package the results
`icvsp_results.zip` appears in the version's **Output** tab. Download it, unzip into `ICVSP/ai/`, and run `python scripts/compare_runs.py` to add the run to the experiment log.

In [ ]:
!cd /kaggle/working/ai && zip -qr /kaggle/working/icvsp_results.zip runs && ls -lh /kaggle/working/icvsp_results.zip